In [ ]:
import csv
import torch
from sentence_transformers import SentenceTransformer
import sys
sys.path.append("/workspaces/hallucilation_in_llm")
from model.hf_model import HFModel

from pipeline.runner import PipelineRunner
from evaluation.evaluator import Evaluator
from decision.final_score import FinalScore
from decision.hallucination_decider import HallucinationDecider
from uncertainty.black_uncertainty import BlackBoxUncertainty
from uncertainty.graybox_uncertainty import GrayBoxUncertainty
from uncertainty.whitebox_uncertainty import WhiteBoxUncertainty
from uncertainty.semantic_uncertainty import EnsembleSemanticUncertainty
from stat_metrics import StatisticalAnalyzer
from calibration import CalibrationMetrics



prompts_tr = [
    "Fransa'nın başkenti neresidir?",
    "Fotosentezi açıklayın.",
    "Mona Lisa'yı kim yaptı?",
    "Kara delikleri tanımlayın.",
    "Kuantum dolanıklığı nedir?"
]

ground_truth_tr = [
    "Paris",
    "Fotosentez, bitkilerin ışığı kimyasal enerjiye dönüştürdüğü süreçtir.",
    "Leonardo da Vinci",
    "Kara delik, hiçbir şeyin kaçamadığı kadar güçlü yerçekimine sahip uzay bölgesidir.",
    "Kuantum dolanıklığı, parçacıkların mesafeye bakılmaksızın birbirleriyle ilişkili olduğu bir fenomendir."
]



model = HFModel("gpt2")  # Örnek: yerel GPT2

uncertainty_modules = {
    "blackbox": lambda output: BlackBoxUncertainty(output.responses),
    "graybox": lambda output: GrayBoxUncertainty(output.responses, output.log_probs),
    "whitebox": lambda output: WhiteBoxUncertainty(output.logits, output.token_ids, output.responses),
    "semantic": lambda output: EnsembleSemanticUncertainty(output.responses, language="tr")
}

final_score_calc = FinalScore()
evaluator = Evaluator(final_score_calc)
decider = HallucinationDecider(thresholds={"hallucination": 0.7})

pipeline = PipelineRunner(model, uncertainty_modules, evaluator, decider)


results = []

for prompt in prompts_tr:
    res = pipeline.run(prompt)
    results.append(res)
    print(f"Prompt: {prompt}")
    print(f"Responses: {res['responses']}")
    print(f"Uncertainty metrics: {res['uncertainty']}")
    print(f"Final evaluation: {res['evaluation']}")
    print(f"Decision: {res['decision']}")
    print("="*50)


class GroundTruthSemanticEvaluator:
    def __init__(self, ground_truth, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=None):
        self.ground_truth = ground_truth
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = SentenceTransformer(model_name, device=self.device)

    def compute_similarity(self, responses):
        sim_scores = []
        for gt_text, resp_list in zip(self.ground_truth, responses):
            gt_emb = self.model.encode(gt_text, convert_to_tensor=True, normalize_embeddings=True)
            resp_embs = self.model.encode(resp_list, convert_to_tensor=True, normalize_embeddings=True)
            sims = torch.nn.functional.cosine_similarity(resp_embs, gt_emb.unsqueeze(0))
            sim_scores.append(float(sims.mean()))
        return sim_scores

responses_list = [r['responses'] for r in results]
gt_eval = GroundTruthSemanticEvaluator(ground_truth_tr)
similarity_scores = gt_eval.compute_similarity(responses_list)

final_scores = [r['evaluation']['final_score'] for r in results]

# Pearson ve Spearman
pearson, spearman = StatisticalAnalyzer.correlation(final_scores, similarity_scores)

# AUROC ve PR-AUC (StatisticalAnalyzer sınıfı içinde binary dönüşüm yapıyor)
auroc = StatisticalAnalyzer.auroc(final_scores, similarity_scores, threshold=0.5)
pr_auc = StatisticalAnalyzer.pr_auc(final_scores, similarity_scores, threshold=0.5)

# Brier ve ECE
brier = CalibrationMetrics.brier_score(final_scores, similarity_scores)
ece = CalibrationMetrics.expected_calibration_error(final_scores, similarity_scores)


csv_columns = [
    "Prompt",
    "Responses",
    "Blackbox_Uncertainty",
    "Graybox_Uncertainty",
    "Whitebox_Uncertainty",
    "Semantic_Uncertainty",
    "Final_Score",
    "Decision",
    "Semantic_Similarity",
    "Pearson_Corr",
    "Spearman_Corr",
    "AUROC",
    "PR_AUC",
    "Brier_Score",
    "ECE"
]

csv_file = "pipeline_results_tr.csv"
with open(csv_file, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_columns)
    writer.writeheader()
    for i, r in enumerate(results):
        writer.writerow({
            "Prompt": prompts_tr[i],
            "Responses": "; ".join(r['responses']),
            "Blackbox_Uncertainty": r['uncertainty'].get('blackbox', None),
            "Graybox_Uncertainty": r['uncertainty'].get('graybox', None),
            "Whitebox_Uncertainty": r['uncertainty'].get('whitebox', None),
            "Semantic_Uncertainty": r['uncertainty'].get('semantic', None),
            "Final_Score": r['evaluation']['final_score'],
            "Decision": r['decision'],
            "Semantic_Similarity": similarity_scores[i],
            "Pearson_Corr": pearson,
            "Spearman_Corr": spearman,
            "AUROC": auroc,
            "PR_AUC": pr_auc,
            "Brier_Score": brier,
            "ECE": ece
        })

print(f"Tüm sonuçlar '{csv_file}' dosyasına kaydedildi.")


/workspaces/hallucilation_in_llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 406.71it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
/workspaces/hallucilation_in_llm/.venv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(
Loadi

Prompt: Fransa'nın başkenti neresidir?
Responses: ["Fransa'nın başkenti neresidir? (You have arrived and have a seat on a bus.)\n\nNo! (What is that?)\n\nThe train left the station and then stopped in the center of the square.\n\nMuktürküş:\n\n", 'Fransa\'nın başkenti neresidir?\n\n"Sünciğuçarın kışmazın yas müzküniçı. Sünciğur nadırıdın, sünci', "Fransa'nın başkenti neresidir? Eğüyük uğuririn zem çığırül!"]
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.0, 'gray_entropy': np.float64(1.0986122886651097), 'white_entropy': nan, 'white_confidence': 0.0, 'white_consistency': 0.3333333333333333, 'semantic_consistency': 0.6361387411753336, 'uncertainty': 0.36386125882466636}
Final evaluation: {'metrics': {'entropy': nan, 'confidence': 0.0, 'self_consistency': 0.3333333333333333}, 'final_score': nan}
Decision: reliable


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 721.37it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 721.56it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 720.82it/s, Material

Prompt: Fotosentezi açıklayın.
Responses: ['Fotosentezi açıklayın.\n\nKımıldin - kımıldin.\n\nNisar - nekar.\n\nSomir - somir.\n\nYusılız - yusılı', 'Fotosentezi açıklayın. pic.twitter.com/QdW5dKbLXXa — Вилака Скоружа (@Вилака) January 28, 2016\n\nThe', 'Fotosentezi açıklayın. İbrahim Bekut. Nihat Çşmılıç. Pımıpı, A. & Alakın. 2010. The risk of smoking and related health problems in Turkish men. J']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.05548179955504352, 'gray_entropy': np.float64(1.0986122886651097), 'white_entropy': nan, 'white_confidence': 1.1576852873580697e-39, 'white_consistency': 0.3333333333333333, 'semantic_consistency': 0.2747424135605494, 'uncertainty': 0.7252575864394506}
Final evaluation: {'metrics': {'entropy': nan, 'confidence': 0.05548179955504352, 'self_consistency': 0.3333333333333333}, 'final_score': nan}
Decision: reliable


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 748.60it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 687.54it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 722.84it/s, Material

...

Prompt: Kara delikleri tanımlayın.
Responses: ['Kara delikleri tanımlayın.\n\nThe second of my two books, The Book of the Dragon, is based on an older book by P. T. S. Janssen and I have been using the same style since that time. It is a wonderful and complex,', 'Kara delikleri tanımlayın.com\n\nThe country\'s two main broadcasters have come under fire for airing cartoons of the Prophet Muhammad on television and in print in Turkey.\n\n"The cartoons were produced by Turkish channel TVN, which broadcast cartoons of Muhammad," a Twitter post', 'Kara delikleri tanımlayın.\n\n"Dinık kara ogli naman darışımı lalıpılılın ogli yinıdın. Nuzı makıyız']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.04157769235366984, 'gray_entropy': np.float64(1.0986122886651097), 'white_entropy': nan, 'white_confidence': 6.2962717576391004e-46, 'white_consistency': 0.3333333333333333, '

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 688.56it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 728.90it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/LaBSE
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 747.67it/s, Material

Prompt: Kuantum dolanıklığı nedir?
Responses: ['Kuantum dolanıklığı nedir?\n\nThe letter has been deleted.\n\nCopyright © 2018 The Washington Times, LLC. Click here for reprint permission.', 'Kuantum dolanıklığı nedir? — Огут Кучаний (@kalandisti) November 3, 2017\n\nFinnish broadcaster TASS confirmed Friday that the group is planning to stage a terrorist attack in central', 'Kuantum dolanıklığı nedir?\n\nT. Yıldırımımımımımımımımımımımımımımımımımımımımı']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.0, 'gray_entropy': np.float64(1.0986122886651097), 'white_entropy': nan, 'white_confidence': 0.0, 'white_consistency': 0.3333333333333333, 'semantic_consistency': 0.24382346651206416, 'uncertainty': 0.7561765334879358}
Final evaluation: {'metrics': {'entropy': nan, 'confidence': 0.0, 'self_consistency': 0.3333333333333333}, 'final_score': nan}
Decision: reliable


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 716.77it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tüm sonuçlar 'pipeline_results_tr.csv' dosyasına kaydedildi.


In [1]:
!pip install sentence_transformers

In [2]:
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 15.8 MB/s  0:00:20m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 37.9 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 27.5 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 82.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 71.0 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 75.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 27.5 MB/s  0:00:11m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 71.1 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 76.9 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 55.6 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/28

In [2]:
!pip install requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [requests]


In [ ]:
import csv
import torch
from sentence_transformers import SentenceTransformer
import sys
sys.path.append("/workspaces/hallucilation_in_llm")
from model.hf_model import HFModel

from pipeline.runner import PipelineRunner
from evaluation.evaluator import Evaluator
from decision.final_score import FinalScore
from decision.hallucination_decider import HallucinationDecider
from uncertainty.black_uncertainty import BlackBoxUncertainty
from uncertainty.graybox_uncertainty import GrayBoxUncertainty
from uncertainty.whitebox_uncertainty import WhiteBoxUncertainty
from uncertainty.semantic_uncertainty import EnsembleSemanticUncertainty
from stat_metrics import StatisticalAnalyzer
from calibration import CalibrationMetrics

prompts_en = [
    "What is the capital of France?",
    "Explain photosynthesis.",
    "Who painted the Mona Lisa?",
    "Define black holes.",
    "What is quantum entanglement?"
]

ground_truth_en = [
    "Paris",
    "Photosynthesis is the process by which plants convert light into chemical energy.",
    "Leonardo da Vinci",
    "A black hole is a region in space with gravity so strong that nothing can escape.",
    "Quantum entanglement is a phenomenon where particles remain connected regardless of distance."
]


model = HFModel("EleutherAI/pythia-70m")  # Example: local GPT2

uncertainty_modules = {
    "blackbox": lambda output: BlackBoxUncertainty(output.responses),
    "graybox": lambda output: GrayBoxUncertainty(output.responses, output.log_probs),
    "whitebox": lambda output: WhiteBoxUncertainty(output.logits, output.token_ids, output.responses),
    "semantic": lambda output: EnsembleSemanticUncertainty(output.responses, language="en")
}

final_score_calc = FinalScore()
evaluator = Evaluator(final_score_calc)
decider = HallucinationDecider(thresholds={"hallucination": 0.7})

pipeline = PipelineRunner(model, uncertainty_modules, evaluator, decider)


results = []

for prompt in prompts_en:
    res = pipeline.run(prompt)
    results.append(res)
    print(f"Prompt: {prompt}")
    print(f"Responses: {res['responses']}")
    print(f"Uncertainty metrics: {res['uncertainty']}")
    print(f"Final evaluation: {res['evaluation']}")
    print(f"Decision: {res['decision']}")
    print("="*50)


class GroundTruthSemanticEvaluator:
    def __init__(self, ground_truth, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=None):
        self.ground_truth = ground_truth
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = SentenceTransformer(model_name, device=self.device)

    def compute_similarity(self, responses):
        sim_scores = []
        for gt_text, resp_list in zip(self.ground_truth, responses):
            gt_emb = self.model.encode(gt_text, convert_to_tensor=True, normalize_embeddings=True)
            resp_embs = self.model.encode(resp_list, convert_to_tensor=True, normalize_embeddings=True)
            sims = torch.nn.functional.cosine_similarity(resp_embs, gt_emb.unsqueeze(0))
            sim_scores.append(float(sims.mean()))
        return sim_scores

responses_list = [r['responses'] for r in results]
gt_eval = GroundTruthSemanticEvaluator(ground_truth_en)
similarity_scores = gt_eval.compute_similarity(responses_list)


final_scores = [r['evaluation']['final_score'] for r in results]

# Pearson and Spearman correlation
pearson, spearman = StatisticalAnalyzer.correlation(final_scores, similarity_scores)

# AUROC and PR-AUC
auroc = StatisticalAnalyzer.auroc(final_scores, similarity_scores, threshold=0.5)
pr_auc = StatisticalAnalyzer.pr_auc(final_scores, similarity_scores, threshold=0.5)

# Brier score and ECE
brier = CalibrationMetrics.brier_score(final_scores, similarity_scores)
ece = CalibrationMetrics.expected_calibration_error(final_scores, similarity_scores)


csv_columns = [
    "Prompt",
    "Responses",
    "Blackbox_Uncertainty",
    "Graybox_Uncertainty",
    "Whitebox_Uncertainty",
    "Semantic_Uncertainty",
    "Final_Score",
    "Decision",
    "Semantic_Similarity",
    "Pearson_Corr",
    "Spearman_Corr",
    "AUROC",
    "PR_AUC",
    "Brier_Score",
    "ECE"
]

csv_file = "pipeline_results_en.csv"
with open(csv_file, 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_columns)
    writer.writeheader()
    for i, r in enumerate(results):
        writer.writerow({
            "Prompt": prompts_en[i],
            "Responses": "; ".join(r['responses']),
            "Blackbox_Uncertainty": r['uncertainty'].get('blackbox', None),
            "Graybox_Uncertainty": r['uncertainty'].get('graybox', None),
            "Whitebox_Uncertainty": r['uncertainty'].get('whitebox', None),
            "Semantic_Uncertainty": r['uncertainty'].get('semantic', None),
            "Final_Score": r['evaluation']['final_score'],
            "Decision": r['decision'],
            "Semantic_Similarity": similarity_scores[i],
            "Pearson_Corr": pearson,
            "Spearman_Corr": spearman,
            "AUROC": auroc,
            "PR_AUC": pr_auc,
            "Brier_Score": brier,
            "ECE": ece
        })

print(f"All English results saved to '{csv_file}'")


/workspaces/hallucilation_in_llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 707.97it/s, Materializing param=gpt_neox.layers.5.post_attention_layernorm.weight] 
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
/workspaces/hallucilation_in_llm/.venv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(
Loading weights: 100%|██████████| 103/103 [00:02<00:00, 34.71it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+----

Prompt: What is the capital of France?
Responses: ['What is the capital of France?\n\nThe French capital is a place where the French capital is divided into two parts. The first part is the capital of France, which is divided into two parts, the capital of France, which is divided into two parts, the capital of France', 'What is the capital of France?\n\nThe French capital is a place where the people are not alone. The people are not alone. The people are not alone. The people are not alone. The people are not alone. The people are not alone. The people are not alone', 'What is the capital of France?\n\nThe French are the only ones who can afford to pay their taxes. They are the only ones who can afford to pay their taxes. They are the only ones who can afford to pay their taxes. They are the only ones who can afford']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence'

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 685.73it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 731.67it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 596.19it/s, Materializing param=poole

Prompt: Explain photosynthesis.
Responses: ['Explain photosynthesis.\n\nThe first step in the process of converting the water to the water is to convert the water to the water. The water is then converted into a water-soluble salt, which is then added to the water. The salt is then added to', 'Explain photosynthesis.\n\nThe first step in the process of the synthesis of the photosynthesis is to use the photosynthesis process to synthesize the photosynthetic pathway. The synthesis of the photosynthesis pathway is then carried out using the photosynthesis process. The synthesis of the', 'Explain photosynthesis.\n\nThe first step in the process is to understand the structure of the protein in the cell, the structure of the cell, and the cell structure of the cell. This is the structure of the cell, and the cell is the structure of the']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gr

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 715.50it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 622.78it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 691.04it/s, Materializing param=poole

...

Prompt: Define black holes.
Responses: ['Define black holes.\n\nThe black hole is a black hole, with a black hole and a black hole.\n\nThe black hole is a black hole, with a black hole and a black hole.\n\nThe black hole is a black hole, with a', 'Define black holes.\n\nA:\n\nThis is a very simple question.\nThe problem is that the only way to do this is to have a black hole in the background.\nThe only way to do this is to have a black hole in the background', 'Define black holes.\n\nThe first black hole is the first black hole of the universe. The black hole is the first black hole of the universe.\n\nThe black hole is the first black hole of the universe. The black hole is the first black hole of']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.28576938704023036, 'gray_entropy': np.float64(1.0986122886651097), 'white_entropy': nan, 'white_confidence': 0.00045

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 677.66it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 727.56it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 698.27it/s, Materializing param=poole

Prompt: What is quantum entanglement?
Responses: ['What is quantum entanglement?\n\nThe quantum entanglement is a measure of the number of particles in a system, which is the number of particles that can be measured. The number of particles in a system is the number of particles that can be measured.\n\nThe number of', 'What is quantum entanglement?\n\nA:\n\nThe quantum entanglement is a quantum entanglement of the form $A_1\\otimes\\cdots\\otimes A_n$ which is a quantum state. It is the quantum entanglement of the form $A_1\\otimes', 'What is quantum entanglement?\n\nThe quantum entanglement is a measure of how much a quantum system can be. The quantum system can be described by a quantum system, and it can be described by a quantum system.\n\nThe quantum system can be described by a quantum system']
Uncertainty metrics: {'black_consistency': 0.3333333333333333, 'black_entropy': np.float64(1.0986122886651097), 'black_confidence': 0.3333333333333333, 'gray_confidence': 0.24803764137718

Loading weights: 100%|██████████| 199/199 [00:01<00:00, 177.20it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All English results saved to 'pipeline_results_en.csv'


In [2]:
from stat_metrics import StatisticalAnalyzer

scores = [0.1, 0.2, 0.8, 0.9, 0.4, 0.7]
labels = [0, 0, 1, 1, 0, 1]

pearson, spearman = StatisticalAnalyzer.correlation(scores, labels)
auroc = StatisticalAnalyzer.auroc(scores, labels)
pr_auc = StatisticalAnalyzer.pr_auc(scores, labels)

print("Pearson:", pearson)
print("Spearman:", spearman)
print("AUROC:", auroc)
print("PR-AUC:", pr_auc)


Pearson: 0.9372403389139513
Spearman: 0.87831006565368
AUROC: 1.0
PR-AUC: 1.0


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

from uncertainty.whitebox_uncertainty import WhiteBoxUncertainty
from uncertainty.semantic_uncertainty import EnsembleSemanticUncertainty
from evaluation.evaluator import Evaluator
from decision.hallucination_decider import HallucinationDecider





MODEL_NAME = "gpt2"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()




def generate_samples(prompt,
                     num_samples=5,
                     max_new_tokens=50,
                     temperature=0.9,
                     top_p=0.95):

    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    all_scores = []
    all_token_ids = []
    all_texts = []

    for i in range(num_samples):

        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            output_scores=True,
            return_dict_in_generate=True
        )

        generated_ids = output.sequences[0]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

        # scores tuple -> list[Tensor]
        scores = torch.stack(output.scores, dim=0).squeeze(1)

        all_scores.append(scores)
        all_token_ids.append(generated_ids.tolist())
        all_texts.append(generated_text)

    return all_scores, all_token_ids, all_texts





test_prompts = [
    "What is the capital of France?",
    "Who discovered gravity?",
    "Explain quantum mechanics in one sentence.",
    "What is 2 + 2?"
]




for prompt in test_prompts:

    print("\n" + "="*80)
    print("PROMPT:", prompt)
    print("="*80)

    scores, token_ids, responses = generate_samples(
        prompt,
        num_samples=5,
        temperature=0.9
    )

    # -------------------------------------------------
    # PRINT ALL RESPONSES
    # -------------------------------------------------
    print("\n--- MODEL RESPONSES ---")
    for i, r in enumerate(responses):
        print(f"\nSample {i+1}:")
        print(r)



    white = WhiteBoxUncertainty(
        scores=scores,
        token_ids=token_ids,
        text_responses=responses
    )

    white_results = white.compute()

    print("\n--- WHITEBOX METRICS ---")
    for k, v in white_results.items():
        print(f"{k}: {v}")



    semantic = EnsembleSemanticUncertainty(
        responses,
        language="en"
    )

    semantic_results = semantic.compute()

    print("\n--- SEMANTIC METRICS ---")
    for k, v in semantic_results.items():
        print(f"{k}: {v}")


    all_metrics = {}
    all_metrics.update(white_results)
    all_metrics.update(semantic_results)


    evaluator = Evaluator()
    evaluation = evaluator.evaluate(all_metrics)

    print("\n--- EVALUATION ---")
    print(evaluation)



    decider = HallucinationDecider(threshold=0.5)
    decision = decider.decide(evaluation)

    print("\n--- DECISION ---")
    print(decision)

/workspaces/hallucilation_in_llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:03<00:00, 37.42it/s, Materializing param=transformer.wte.weight]              
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



PROMPT: What is the capital of France?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



--- MODEL RESPONSES ---

Sample 1:
What is the capital of France?

France's capital is located in Marseille.

In its capital, Paris, a number of things are very important. Among them is the famous Rennes cathedral in which thousands of pilgrims pass every year. This cathedral was founded on the

Sample 2:
What is the capital of France? It's a little different than Paris. You'd think we could get into the game of being a nation that wasn't a big part of the continent.

So, we have to try and find the right balance between what our culture is about

Sample 3:
What is the capital of France? Well it is called the capital of France. It is also called the capital of Switzerland. But Switzerland is called the capital of the whole world. If you can see all this, then all the other capital is capital of the world. If you want

Sample 4:
What is the capital of France?

The capital of France is Paris, which has only 756,000 inhabitants (1). France is a small country with relatively little immigr

/workspaces/hallucilation_in_llm/.venv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(
Loading weights: 100%|██████████| 103/103 [00:02<00:00, 43.96it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Loading weights: 100%|██████████| 199/199 [00:07<00:00, 27.13it/s, Materializing param=pooler.dense.weight]                         
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading semantic model: sentence-transformers/paraphrase-MiniLM-L12-v2


Loading weights: 100%|██████████| 199/199 [00:05<00:00, 37.14it/s, Materializing param=pooler.dense.weight]                                
BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- SEMANTIC METRICS ---
semantic_consistency: 0.7173886895179749
semantic_uncertainty: 0.28261131048202515

--- EVALUATION ---
{'metrics': {'entropy': 0.0, 'confidence': 0.12453511715498201, 'self_consistency': 0.7173886895179749}, 'final_score': 0.2806412688909856}


TypeError: HallucinationDecider.__init__() got an unexpected keyword argument 'threshold'

In [1]:
import csv
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
import sys

sys.path.append("/workspaces/hallucilation_in_llm")

from model.hf_model import HFModel
from pipeline.runner import PipelineRunner
from evaluation.evaluator import Evaluator
from decision.final_score import FinalScore
from decision.hallucination_decider import HallucinationDecider
from uncertainty.black_uncertainty import BlackBoxUncertainty
from uncertainty.graybox_uncertainty import GrayBoxUncertainty
from uncertainty.whitebox_uncertainty import WhiteBoxUncertainty
from uncertainty.semantic_uncertainty import EnsembleSemanticUncertainty
from stat_metrics import StatisticalAnalyzer
from calibration import CalibrationMetrics





prompts_en = [
    "What is the capital of France?",
    "Explain photosynthesis.",
    "Who painted the Mona Lisa?",
    "Define black holes.",
    "What is quantum entanglement?"
]

ground_truth_en = [
    "Paris",
    "Photosynthesis is the process by which plants convert light into chemical energy.",
    "Leonardo da Vinci",
    "A black hole is a region in space with gravity so strong that nothing can escape.",
    "Quantum entanglement is a phenomenon where particles remain connected regardless of distance."
]




model = HFModel("EleutherAI/pythia-70m")

uncertainty_modules = {
    "blackbox": lambda output: BlackBoxUncertainty(output.responses),
    "graybox": lambda output: GrayBoxUncertainty(output.responses, output.log_probs),
    "whitebox": lambda output: WhiteBoxUncertainty(output.logits, output.token_ids, output.responses),
    "semantic": lambda output: EnsembleSemanticUncertainty(output.responses, language="en")
}

final_score_calc = FinalScore()
evaluator = Evaluator()

decider = HallucinationDecider(
    thresholds={
        "entropy": 5.0,
        "confidence": 0.35,
        "consistency": 0.5,
        "risk": 2.5
    }
)

pipeline = PipelineRunner(model, uncertainty_modules, evaluator, decider)




results = []

for prompt in prompts_en:
    res = pipeline.run(prompt)
    results.append(res)

    print("=" * 60)
    print("PROMPT:", prompt)
    print("RESPONSES:", res["responses"])
    print("UNCERTAINTY:", res["uncertainty"])
    print("FINAL SCORE:", res["evaluation"]["final_score"])
    print("DECISION:", res["decision"])




class GroundTruthSemanticEvaluator:
    def __init__(self, ground_truth, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=None):
        self.ground_truth = ground_truth
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = SentenceTransformer(model_name, device=self.device)

    def compute_similarity(self, responses):
        sim_scores = []
        for gt_text, resp_list in zip(self.ground_truth, responses):

            gt_emb = self.model.encode(gt_text, convert_to_tensor=True, normalize_embeddings=True)
            resp_embs = self.model.encode(resp_list, convert_to_tensor=True, normalize_embeddings=True)

            sims = torch.nn.functional.cosine_similarity(resp_embs, gt_emb.unsqueeze(0))
            sim_scores.append(float(sims.mean()))

        return sim_scores


responses_list = [r["responses"] for r in results]
gt_eval = GroundTruthSemanticEvaluator(ground_truth_en)

similarity_scores = gt_eval.compute_similarity(responses_list)




# similarity > 0.7 => correct (0 = reliable)
# similarity <= 0.7 => hallucination (1 = hallucination)

labels = [1 if sim < 0.7 else 0 for sim in similarity_scores]


final_scores = [r["evaluation"]["final_score"] for r in results]

final_scores = np.array(final_scores)
labels = np.array(labels)



pearson, spearman = StatisticalAnalyzer.correlation(final_scores, similarity_scores)

auroc = StatisticalAnalyzer.auroc(final_scores, labels)
pr_auc = StatisticalAnalyzer.pr_auc(final_scores, labels)

brier = CalibrationMetrics.brier_score(final_scores, labels)
ece = CalibrationMetrics.expected_calibration_error(final_scores, labels)


print("\n===== GLOBAL METRICS =====")
print("Pearson:", pearson)
print("Spearman:", spearman)
print("AUROC:", auroc)
print("PR-AUC:", pr_auc)
print("Brier Score:", brier)
print("ECE:", ece)




csv_columns = [
    "Prompt",
    "Responses",
    "Blackbox",
    "Graybox",
    "Whitebox",
    "Semantic",
    "Final_Score",
    "Decision",
    "Semantic_Similarity",
    "Label",
    "Pearson",
    "Spearman",
    "AUROC",
    "PR_AUC",
    "Brier",
    "ECE"
]

csv_file = "pipeline_results_en.csv"

with open(csv_file, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_columns)
    writer.writeheader()

    for i, r in enumerate(results):
        writer.writerow({
            "Prompt": prompts_en[i],
            "Responses": " || ".join(r["responses"]),
            "Blackbox": r["uncertainty"].get("blackbox"),
            "Graybox": r["uncertainty"].get("graybox"),
            "Whitebox": r["uncertainty"].get("whitebox"),
            "Semantic": r["uncertainty"].get("semantic"),
            "Final_Score": final_scores[i],
            "Decision": r["decision"],
            "Semantic_Similarity": similarity_scores[i],
            "Label": labels[i],
            "Pearson": pearson,
            "Spearman": spearman,
            "AUROC": auroc,
            "PR_AUC": pr_auc,
            "Brier": brier,
            "ECE": ece
        })

print(f"\nAll results saved to '{csv_file}'")

/workspaces/hallucilation_in_llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 703.63it/s, Materializing param=gpt_neox.layers.5.post_attention_layernorm.weight] 
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Loading semantic model: sentence-transformers/all-MiniLM-L6-v2


/workspaces/hallucilation_in_llm/.venv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 140.99it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading semantic model: sentence-transformers/all-mpnet-base-v2


Loading weights: 100%|██████████| 199/199 [00:01<00:00, 196.42it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

PROMPT: Who painted the Mona Lisa?
RESPONSES: ['Who painted the Mona Lisa? I think it was the same.\n\nIt was a new world, no one. I was a bit young and I knew it.\n\nNo, it was not a new world, no one had a new idea, it was a new', 'Who painted the Mona Lisa?\n\nThe story takes the form of an abstract painting that has been commissioned by the author, for which he is charged to be a part of his creation, and which has been published in the New York Times.\n\nIn his book, The', 'Who painted the Mona Lisa?\n\nI would like to thank the creators for their effort and dedication.\n\nThe new album “The Life of the Mona Lisa” was released on June 3, 2018. The album was also a tribute to the original guitarist for the album', 'Who painted the Mona Lisa?\n\nThe only one who gets a lot of attention is the two-bedroom one. The kitchen was empty and the living room was closed, but the room had to be cleaned up. The kitchen was empty and the living room was lit.', 'Who painted the Mona Lisa? How ab

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


PROMPT: Define black holes.
RESPONSES: ['Define black holes.\n\n**W** = **G**\n\n# Contents\n\n1. Cover\n\n1. Title Page\n\n1. Copyright\n\n2. Dedication\n\nChapter 1\n\nChapter 2\n\nChapter 3\n\nChapter', 'Define black holes.\n\n#include <windows.h>\n#include "WinMain.h"\n\n#if (WIN32)\n    winMain.h\n    winMain.h\n    winMain.h\n    winMain.h', 'Define black holes.\n\nI know what this is about, but you could probably use something like the following code:\nusing System;\nusing System.Collections.Generic;\nusing System.Linq;\nusing System.Threading.Tasks;\n\n', 'Define black holes.\n\nAt first, the lightest elements of the object were found on the horizon. The first one was placed on the right and the second one was placed on the left and the third one was placed on the right. The second one was placed', 'Define black holes.\n\nA:\n\nI suspect your problem is because the problem is very much in the way that the black hole is a black hole, but the hole is not a hole to be black. The i

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 245.29it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===== GLOBAL METRICS =====
Pearson: 0.0
Spearman: 0.0
AUROC: 0.5
PR-AUC: 0.9
Brier Score: 0.8
ECE: 0.8

All results saved to 'pipeline_results_en.csv'


In [3]:
from uncertainty.whitebox_uncertainty import WhiteBoxUncertainty

import torch

# 1 sample, 3 token üretmiş, vocab=5
scores = [
    torch.tensor([
        [2.0, 0.5, 0.1, -1.0, 0.0],   # token 1 logits
        [1.2, 2.5, -0.3, 0.0, -1.0],  # token 2 logits
        [0.1, -0.2, 3.0, 0.5, -0.7]   # token 3 logits
    ])
]

token_ids = [
    [0, 1, 2]  # seçilen token indexleri
]


text_responses = [
    "Paris is the capital of France."
]

wb = WhiteBoxUncertainty(scores, token_ids, text_responses)

print(wb.compute())


KeyboardInterrupt: 

In [2]:
from decision.final_score import FinalScore
from decision.hallucination_decider import HallucinationDecider
import numpy as np

thresholds = {
    "entropy": 2.0,
    "confidence": 0.3,
    "consistency": 0.3,
    "risk": 0.5
}

metrics = {
    'whitebox_white_entropy': np.float64(1.471),
    'whitebox_white_confidence': np.float64(0.216),
    'whitebox_white_consistency': np.float64(0.2),
    'semantic_semantic_consistency': np.float64(0.55)
}

fs = FinalScore()
final_score = fs.compute(metrics)

evaluation_scores = {
    "metrics": {
        "entropy": metrics['whitebox_white_entropy'],
        "confidence": metrics['whitebox_white_confidence'],
        "consistency": metrics['whitebox_white_consistency'],
        "semantic": metrics['semantic_semantic_consistency']
    },
    "final_score": final_score
}

decider = HallucinationDecider(thresholds)
decision = decider.decide(evaluation_scores)

print("Final Score:", final_score)  # artık 0 değil ~0.6 çıkmalı
print("Decision:", decision)        # "hallucination" veya "reliable"

Final Score: 0.56647
Decision: hallucination


In [ ]:
# Gerekli kütüphaneler
import torch
import numpy as np
import math
from transformers import GPT2LMHeadModel, GPT2Tokenizer



def safe_float(value):
    """NumPy ve Python float tiplerini güvenli float'a çevirir"""
    if value is None:
        return 0.0
    if isinstance(value, (float, np.floating)):
        if math.isnan(value) or math.isinf(value):
            return 0.0
        return float(value)
    try:
        return float(value)
    except:
        return 0.0



class FinalScore:
    def __init__(self, entropy_max=5.0):
        self.entropy_max = entropy_max

    def compute(self, metrics: dict):
        entropy = safe_float(metrics.get("whitebox_white_entropy", 0.0))
        confidence = safe_float(metrics.get("whitebox_white_confidence", 1.0))
        consistency = safe_float(metrics.get("whitebox_white_consistency", 1.0))
        semantic = safe_float(metrics.get("semantic_semantic_consistency", 1.0))

        entropy_norm = min(entropy / self.entropy_max, 1.0)

        uncertainty_score = (
            0.35 * entropy_norm +
            0.25 * (1 - consistency) +
            0.25 * (1 - confidence) +
            0.15 * (1 - semantic)
        )
        return float(uncertainty_score)



class HallucinationDecider:
    def __init__(self, thresholds):
        self.thresholds = thresholds

    def decide(self, evaluation_scores):
        metrics = evaluation_scores.get("metrics", {})
        final_score = safe_float(evaluation_scores.get("final_score", 0.0))

        entropy = safe_float(metrics.get("entropy"))
        confidence = safe_float(metrics.get("confidence"))
        consistency = safe_float(metrics.get("consistency"))

        # Basit threshold kararları
        if entropy > self.thresholds.get("entropy", float("inf")):
            return "hallucination"
        if confidence < self.thresholds.get("confidence", 0.0):
            return "hallucination"
        if consistency < self.thresholds.get("consistency", 0.0):
            return "hallucination"

        # Risk bazlı
        risk = 0.4 * entropy + 0.3 * (1 - confidence) + 0.3 * (1 - consistency)
        if risk > self.thresholds.get("risk", 1.0):
            return "hallucination"

        return "reliable"



# Model ve tokenizer yükle
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()

# Test prompt
prompt = "What is the capital of France?"

# Tokenize
input_ids = tokenizer.encode(prompt, return_tensors="pt")

# Text generation
with torch.no_grad():
    outputs = model.generate(
    input_ids,
    max_length=50,
    num_return_sequences=3,
    do_sample=True,      # sampling açıldı
    top_p=0.9,           # nucleus sampling
    top_k=50,            # top-k sampling
    pad_token_id=tokenizer.eos_token_id,
    repetition_penalty=1.2  # tekrar eden token’ları azaltır
)
# Üretilen cevaplar
responses = [tokenizer.decode(out, skip_special_tokens=True) for out in outputs]
print("RESPONSES:")
for r in responses:
    print("-", r)



# GPT-2 çıktısı üzerinden örnek metric hesaplayalım (dummy değerler)
# Not: gerçek modelde bunlar log-prob, entropy vs olabilir
UNCERTAINTY = {
    "whitebox_white_entropy": np.float64(1.2),
    "whitebox_white_confidence": np.float64(0.3),
    "whitebox_white_consistency": np.float64(0.25),
    "semantic_semantic_consistency": np.float64(0.6)
}



fs = FinalScore()
final_score = fs.compute(UNCERTAINTY)

evaluation_scores = {
    "metrics": {
        "entropy": UNCERTAINTY["whitebox_white_entropy"],
        "confidence": UNCERTAINTY["whitebox_white_confidence"],
        "consistency": UNCERTAINTY["whitebox_white_consistency"],
        "semantic": UNCERTAINTY["semantic_semantic_consistency"]
    },
    "final_score": final_score
}

thresholds = {
    "entropy": 2.0,
    "confidence": 0.3,
    "consistency": 0.3,
    "risk": 0.5
}

decider = HallucinationDecider(thresholds)
decision = decider.decide(evaluation_scores)

print("\nFINAL SCORE:", final_score)
print("DECISION:", decision)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 684.88it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RESPONSES:
- What is the capital of France?
The French republic has a population that's 6.2 million people, less than half what it was when Napoleon started his war in 1809 or 19th century Paris to be exact; an estimated 200% higher
- What is the capital of France?
Laurieux, Switzerland. As part in its ongoing bid for independence from Britain after WW2, Italy has now taken a seat on NATO and joined Germany as an active member with new members expected next year
- What is the capital of France?
In this question, we can see that in January 2012 (when Hollande became president), at least 40 per cent—and presumably more than 60 million people around Europe and Latin America —of its population lives within five

FINAL SCORE: 0.5065
DECISION: hallucination


In [1]:
# %%
import torch
import numpy as np
from itertools import combinations
from sentence_transformers import SentenceTransformer
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

# ------------------------------
# ENSEMBLE SEMANTIC UNCERTAINTY
# ------------------------------
class EnsembleSemanticUncertainty:
    EN_MODELS = [
        "sentence-transformers/all-MiniLM-L6-v2",
        "sentence-transformers/all-mpnet-base-v2",
        "sentence-transformers/paraphrase-MiniLM-L12-v2",
    ]
    _MODEL_CACHE = {}

    def __init__(self, responses, device=None):
        self.responses = responses
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.encoders = []

        for name in self.EN_MODELS:
            if name not in self._MODEL_CACHE:
                print(f"Loading semantic model: {name}")
                self._MODEL_CACHE[name] = SentenceTransformer(name, device=self.device)
            self.encoders.append(self._MODEL_CACHE[name])

    def _semantic_consistency_single_model(self, encoder):
        if len(self.responses) < 2:
            return 1.0
        with torch.no_grad():
            embeddings = encoder.encode(
                self.responses, convert_to_tensor=True, normalize_embeddings=True
            )
        sims = []
        for i, j in combinations(range(len(embeddings)), 2):
            sim = torch.nn.functional.cosine_similarity(
                embeddings[i].unsqueeze(0), embeddings[j].unsqueeze(0)
            ).item()
            sims.append(sim)
        return float(np.mean(sims)) if sims else 1.0

    def compute(self):
        scores = [self._semantic_consistency_single_model(enc) for enc in self.encoders]
        consistency = float(np.mean(scores))
        return {
            "semantic_consistency": consistency,
            "semantic_uncertainty": 1 - consistency
        }

# -----------------------------------
# GROUND TRUTH SEMANTIC EVALUATOR
# -----------------------------------
class GroundTruthSemanticEvaluator:
    def __init__(self, ground_truth, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=None):
        self.ground_truth = ground_truth
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = SentenceTransformer(model_name, device=self.device)

    def compute_similarity(self, responses):
        sim_scores = []
        for gt_text, resp_list in zip(self.ground_truth, responses):
            gt_emb = self.model.encode(gt_text, convert_to_tensor=True, normalize_embeddings=True)
            resp_embs = self.model.encode(resp_list, convert_to_tensor=True, normalize_embeddings=True)
            sims = torch.nn.functional.cosine_similarity(resp_embs, gt_emb.unsqueeze(0))
            sim_scores.append(float(sims.mean()))
        return sim_scores

# -------------------
# GPT2 MODEL SETUP
# -------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "gpt2"
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
model.eval()

# -------------------
# PROMPTS & GROUND TRUTH
# -------------------
prompts = [
    "What is the capital of France?",
    "Explain photosynthesis.",
    "Who painted the Mona Lisa?",
    "Define black holes.",
    "What is quantum entanglement?"
]

ground_truth = [
    "Paris",
    "Photosynthesis is the process by which plants convert light into chemical energy.",
    "Leonardo da Vinci",
    "A black hole is a region in space with gravity so strong that nothing can escape.",
    "Quantum entanglement is a phenomenon where particles remain connected regardless of distance."
]

# -------------------
# HELPER FUNCTIONS
# -------------------
def generate_responses(prompt, n_responses=3, max_len=50):
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    outputs = model.generate(
        input_ids,
        do_sample=True,
        temperature=0.8,
        top_k=50,
        top_p=0.95,
        max_length=max_len,
        num_return_sequences=n_responses,
        pad_token_id=tokenizer.eos_token_id
    )
    responses = [tokenizer.decode(out, skip_special_tokens=True) for out in outputs]
    return responses

def compute_final_score(semantic_consistency, semantic_uncertainty):
    # Basit örnek: final score = 1 - uncertainty
    return semantic_consistency

def hallucination_decision(final_score, threshold=0.7):
    # 0 = doğru, 1 = hallucination
    return 0 if final_score >= threshold else 1

# -------------------
# RUN TEST
# -------------------
all_results = []
for prompt in prompts:
    responses = generate_responses(prompt, n_responses=3)
    ensemble = EnsembleSemanticUncertainty(responses)
    sem_unc = ensemble.compute()
    final_score = compute_final_score(sem_unc["semantic_consistency"], sem_unc["semantic_uncertainty"])
    decision = hallucination_decision(final_score, threshold=0.7)

    all_results.append({
        "prompt": prompt,
        "responses": responses,
        "semantic_consistency": sem_unc["semantic_consistency"],
        "semantic_uncertainty": sem_unc["semantic_uncertainty"],
        "final_score": final_score,
        "decision": decision
    })

    print("="*60)
    print("PROMPT:", prompt)
    print("RESPONSES:", responses)
    print("SEMANTIC CONSISTENCY:", sem_unc["semantic_consistency"])
    print("SEMANTIC UNCERTAINTY:", sem_unc["semantic_uncertainty"])
    print("FINAL SCORE:", final_score)
    print("DECISION:", "HALLUCINATION" if decision else "CORRECT")

/workspaces/hallucilation_in_llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:04<00:00, 31.25it/s, Materializing param=transformer.wte.weight]             
GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Loading semantic model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:02<00:00, 35.56it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading semantic model: sentence-transformers/all-mpnet-base-v2


Loading weights: 100%|██████████| 199/199 [00:06<00:00, 29.25it/s, Materializing param=pooler.dense.weight]                         
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading semantic model: sentence-transformers/paraphrase-MiniLM-L12-v2


Loading weights: 100%|██████████| 199/199 [00:05<00:00, 36.54it/s, Materializing param=pooler.dense.weight]                                
BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PROMPT: What is the capital of France?
RESPONSES: ['What is the capital of France?\n\nThe capital of France is Paris, and it is only by using this capital that we can make our country more beautiful.\n\nI am very much in favour of this capital. I think that we can', 'What is the capital of France? A French capital.\n\n"Capital is a symbol of how far the French can push back against the French Empire," said René Lefebvre, professor of French at the University of Strasbourg.\n', 'What is the capital of France?\n\nThe capital of France is the capital of France. It is the capital of France.\n\nWhat is the capital of France?\n\nThe capital of France is the capital of France. It is the']
SEMANTIC CONSISTENCY: 0.7831788725323147
SEMANTIC UNCERTAINTY: 0.21682112746768534
FINAL SCORE: 0.7831788725323147
DECISION: CORRECT
PROMPT: Explain photosynthesis.
RESPONSES: ['Explain photosynthesis.\n\nThe world of plants and animals is largely made up of trees and other small organisms. The leaves of man

In [1]:
# %%
import csv
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
import sys

# Kendi proje klasörünü path’e ekle
sys.path.append("/workspaces/hallucilation_in_llm")

# LLM pipeline importları
from model.hf_model import HFModel
from pipeline.runner import PipelineRunner
from evaluation.evaluator import Evaluator
from decision.final_score import FinalScore
from decision.hallucination_decider import HallucinationDecider
from uncertainty.black_uncertainty import BlackBoxUncertainty
from uncertainty.graybox_uncertainty import GrayBoxUncertainty
from uncertainty.whitebox_uncertainty import WhiteBoxUncertainty
from uncertainty.semantic_uncertainty import EnsembleSemanticUncertainty
from stat_metrics import StatisticalAnalyzer
from calibration import CalibrationMetrics

# ----------------------
# PROMPT & GROUND TRUTH
# ----------------------
prompts_en = [
    "What is the capital of France?",
    "Explain photosynthesis.",
    "Who painted the Mona Lisa?",
    "Define black holes.",
    "What is quantum entanglement?"
]

ground_truth_en = [
    "Paris",
    "Photosynthesis is the process by which plants convert light into chemical energy.",
    "Leonardo da Vinci",
    "A black hole is a region in space with gravity so strong that nothing can escape.",
    "Quantum entanglement is a phenomenon where particles remain connected regardless of distance."
]

# ----------------------
# MODEL & PIPELINE SETUP
# ----------------------
model = HFModel("EleutherAI/pythia-70m")

uncertainty_modules = {
    "blackbox": lambda output: BlackBoxUncertainty(output.responses),
    "graybox": lambda output: GrayBoxUncertainty(output.responses, output.log_probs),
    "whitebox": lambda output: WhiteBoxUncertainty(output.logits, output.token_ids, output.responses),
    "semantic": lambda output: EnsembleSemanticUncertainty(output.responses, language="en")
}

final_score_calc = FinalScore()
evaluator = Evaluator()

decider = HallucinationDecider(
    thresholds={
        "entropy": 5.0,
        "confidence": 0.35,
        "consistency": 0.5,
        "risk": 2.5
    }
)

pipeline = PipelineRunner(model, uncertainty_modules, evaluator, decider)

# ----------------------
# RUN PIPELINE
# ----------------------
results = []
for prompt in prompts_en:
    res = pipeline.run(prompt)
    results.append(res)
    print("="*60)
    print("PROMPT:", prompt)
    print("RESPONSES:", res["responses"])
    print("UNCERTAINTY:", res["uncertainty"])
    print("FINAL SCORE:", res["evaluation"]["final_score"])
    print("DECISION:", res["decision"])

# ----------------------
# GROUND TRUTH SEMANTIC EVALUATION
# ----------------------
class GroundTruthSemanticEvaluator:
    def __init__(self, ground_truth, model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=None):
        self.ground_truth = ground_truth
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = SentenceTransformer(model_name, device=self.device)

    def compute_similarity(self, responses):
        sim_scores = []
        for gt_text, resp_list in zip(self.ground_truth, responses):
            gt_emb = self.model.encode(gt_text, convert_to_tensor=True, normalize_embeddings=True)
            resp_embs = self.model.encode(resp_list, convert_to_tensor=True, normalize_embeddings=True)
            sims = torch.nn.functional.cosine_similarity(resp_embs, gt_emb.unsqueeze(0))
            sim_scores.append(float(sims.mean()))
        return sim_scores

responses_list = [r["responses"] for r in results]
gt_eval = GroundTruthSemanticEvaluator(ground_truth_en)
similarity_scores = gt_eval.compute_similarity(responses_list)

# Labels: similarity > 0.7 -> correct, else hallucination
labels = [1 if sim < 0.7 else 0 for sim in similarity_scores]

final_scores = np.array([r["evaluation"]["final_score"] for r in results])
labels = np.array(labels)

# ----------------------
# GLOBAL METRICS
# ----------------------
pearson, spearman = StatisticalAnalyzer.correlation(final_scores, similarity_scores)
auroc = StatisticalAnalyzer.auroc(final_scores, labels)
pr_auc = StatisticalAnalyzer.pr_auc(final_scores, labels)
brier = CalibrationMetrics.brier_score(final_scores, labels)
ece = CalibrationMetrics.expected_calibration_error(final_scores, labels)

print("\n===== GLOBAL METRICS =====")
print("Pearson:", pearson)
print("Spearman:", spearman)
print("AUROC:", auroc)
print("PR-AUC:", pr_auc)
print("Brier Score:", brier)
print("ECE:", ece)

# ----------------------
# SAVE TO CSV
# ----------------------
csv_columns = [
    "Prompt",
    "Responses",
    "Blackbox",
    "Graybox",
    "Whitebox",
    "Semantic",
    "Final_Score",
    "Decision",
    "Semantic_Similarity",
    "Label",
    "Pearson",
    "Spearman",
    "AUROC",
    "PR_AUC",
    "Brier",
    "ECE"
]

csv_file = "pipeline_results_en.csv"

with open(csv_file, "w", newline="", encoding="utf-8") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_columns)
    writer.writeheader()

    for i, r in enumerate(results):
        writer.writerow({
            "Prompt": prompts_en[i],
            "Responses": " || ".join(r["responses"]),
            "Blackbox": r["uncertainty"].get("blackbox"),
            "Graybox": r["uncertainty"].get("graybox"),
            "Whitebox": r["uncertainty"].get("whitebox"),
            "Semantic": r["uncertainty"].get("semantic"),
            "Final_Score": final_scores[i],
            "Decision": r["decision"],
            "Semantic_Similarity": similarity_scores[i],
            "Label": labels[i],
            "Pearson": pearson,
            "Spearman": spearman,
            "AUROC": auroc,
            "PR_AUC": pr_auc,
            "Brier": brier,
            "ECE": ece
        })

print(f"\nAll results saved to '{csv_file}'")

/workspaces/hallucilation_in_llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 76/76 [00:01<00:00, 63.20it/s, Materializing param=gpt_neox.layers.5.post_attention_layernorm.weight] 
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Loading semantic model: sentence-transformers/all-MiniLM-L6-v2


/workspaces/hallucilation_in_llm/.venv/lib/python3.12/site-packages/sentence_transformers/SentenceTransformer.py:204: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v4 of SentenceTransformers.
  warnings.warn(
Loading weights: 100%|██████████| 103/103 [00:02<00:00, 41.87it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading semantic model: sentence-transformers/all-mpnet-base-v2


Loading weights: 100%|██████████| 199/199 [00:06<00:00, 30.22it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

PROMPT: Who painted the Mona Lisa?
RESPONSES: [" It's the first time I've seen a guy with a red hat with a blue hat, a red hat, and a white hat. I'll be there with the purple hat, right and left and right as a white hat. I'm really", '\n\nThe man was wearing a headscarred jacket and tie, his face a little pale with some redness.\n\n"They must be the men of the Mona Lisa."\n\n"They must have taken the two men with them', '\n\nWhen it comes to the red-colored, blue-blond, white-blond, black-blond, white-blond, black-blond, white-blond, white-blond, black-blond', ' How did it make me even look like you were going to be a child?\n\nWorse, I was just wearing a skirt that looked really good. I said I was going to wear that skirt. I said it was a red skirt,', "\n\nI got the picture and my boyfriend took it out, that's how he left it.\n\n1. How he got it from the hand on the wall.\n\n2. How he got it from the hand on the wall."]
UNCERTAINTY: {'blackbox_black_consistency': 0.2, 'blackbox_black_

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


PROMPT: Define black holes.
RESPONSES: ['\n\nA:\n\nAs mentioned in the answer, the question is,\n\n"The first is the first black hole."\n\nIn the answer, the question "How can we find black holes?"\n\nThis answer is based on', '\n\nA:\n\nA:\n\nThe solution is not to be a function of a given object, but it is a well-known function, that can be defined by a function of a type. This is often used to check the', ' The $L$ are the $T$-periodic Hall-averaged spin functions and the $L$-periodic Hall-averaged spin functions are generated by the magnetic field. For simplicity we will include here only the $L$ in the $T', "\n\nA:\n\nLet's see what happens.\nIn your code, you should be able to have a small number of holes in the left side, which can be accessed through a few holes in the right side of the code.\n", ' But the black holes are already in the dark, so you can’t expect them to be as dark as they come. So if you don’t know what this is, look at the black hole and look at it as dark as 

Loading weights: 100%|██████████| 199/199 [00:04<00:00, 46.66it/s, Materializing param=pooler.dense.weight]                                
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



===== GLOBAL METRICS =====
Pearson: -0.7400252687218587
Spearman: -0.8999999999999998
AUROC: 0.5
PR-AUC: 0.0
Brier Score: 0.11734507355293149
ECE: 0.3397087833015802

All results saved to 'pipeline_results_en.csv'
